In [1]:
import numpy as np
import pandas as pd
import statistics as st
import re
import csv
import scanpy as sc 
import matplotlib
from matplotlib import pyplot as plt
from scipy.stats import binom
from scipy.stats import multinomial
import seaborn
from scipy.stats import hypergeom
import warnings
warnings.filterwarnings('ignore')

import json
print("packages loaded")

packages loaded


In [2]:
adata = sc.read("/project/GCRB/Hon_lab/s215194/Single_Cell/EP1/clonal_cell_removal/pipeline_output/clonal_cell_removal_transcriptome.h5ad", backed='r')


In [3]:
adata

AnnData object with n_obs × n_vars = 357246 × 34395 backed at '/project/GCRB/Hon_lab/s215194/Single_Cell/EP1/clonal_cell_removal/pipeline_output/clonal_cell_removal_transcriptome.h5ad'
    obs: 'percent_RP', 'n_counts_RP', 'n_counts_all', 'louvain'
    var: 'gene_ids', 'feature_types', 'genome', 'n_counts', 'mean', 'std'
    uns: 'log1p', 'louvain', 'louvain_colors', 'louvain_sizes', 'neighbors', 'paga', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [4]:
adata.obs

,percent_RP,n_counts_RP,n_counts_all,louvain
AAACCTGGTCTCCACT-16,0.077775,4423.0,3576.0,3
AAAGATGCAAGTCTGT-9,0.044348,1150.0,1018.0,2
AAAGTAGAGAGTCGGT-2,0.046218,4349.0,3933.0,4
AAAGTAGGTTCGCGAC-9,0.030312,3266.0,3041.0,0
AACACGTAGTGAATTG-7,0.047753,5696.0,5028.0,5
...,...,...,...,...
TGTGTTTGTCACAAGG-4,0.042459,2944.0,2663.0,0
TTCTCAATCTCCTATA-2,0.032038,1904.0,1752.0,0
TTGACTTCACTATCTT-15,0.076482,4184.0,3413.0,5
TTGACTTTCCGAAGAG-12,0.104539,3635.0,2793.0,1


In [5]:
adata.var

,gene_ids,feature_types,genome,n_counts,mean,std
MIR1302-2HG,ENSG00000243485,Gene Expression,GRCh38,213.0,0.000321,0.015859
AL627309.1,ENSG00000238009,Gene Expression,GRCh38,2082.0,0.002813,0.044671
AL627309.3,ENSG00000239945,Gene Expression,GRCh38,1.0,0.000002,0.001043
AL627309.2,ENSG00000239906,Gene Expression,GRCh38,52.0,0.000070,0.007094
AL627309.5,ENSG00000241860,Gene Expression,GRCh38,9254.0,0.012441,0.093373
...,...,...,...,...,...,...
AC136616.2,ENSG00000277761,Gene Expression,GRCh38,2.0,0.000003,0.001671
AC023491.2,ENSG00000278633,Gene Expression,GRCh38,4.0,0.000006,0.002103
AC007325.1,ENSG00000276017,Gene Expression,GRCh38,2.0,0.000002,0.000877
AC007325.4,ENSG00000278817,Gene Expression,GRCh38,2181.0,0.003006,0.046107


In [6]:
adata_singlets = pd.read_pickle('/project/GCRB/Hon_lab/s215194/Single_Cell/EP1/_all_lanes_combined/aggr_dataframe/aggr_combined_df_full.pkl')

In [7]:
adata_singlets = adata_singlets.T

In [8]:
adata_singlets

,BEX3_1_ctl,BEX3_2_ctl,BEX3_3_ctl,BEX3_4_ctl,FN1_1_1_trunc_ctl,FN1_1_2_trunc_ctl,FN1_1_3_trunc_ctl,FN1_2_1_trunc_ctl,FN1_2_3_trunc_ctl,FN1_pro_1_1_ctl,...,chrX_67546350_G_C_AR_polyalanine_3,chrX_67546350_G_C_AR_polyalanine_4,chrX_67546350_G_C_AR_polyalanine_ref_1,chrX_67546350_G_C_AR_polyalanine_ref_2,chrX_67546351_C_T_AR_polyalanine_1,chrX_67546351_C_T_AR_polyalanine_2,chrX_67546351_C_T_AR_polyalanine_3,chrX_67546351_C_T_AR_polyalanine_4,chrX_67546351_C_T_AR_polyalanine_ref_1,chrX_67546351_C_T_AR_polyalanine_ref_2
AAACCTGAGAAGGACA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCTGAGACTAGAT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCTGAGCGTTCCG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCTGAGGAGTACC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCTGAGGCGTACA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GTTGTGCAGGGATAGT-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GTTGTGGGTAACTCGG-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GTTGTGGGTAGTCGCA-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GTTGTGGGTATTGTCT-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# --- 2. Extract the index values from adata.obs ---
# This gets the index of the observations (cells) from your 'adata' object.
desired_indices = adata.obs.index

In [10]:
desired_indices

Index(['AAACCTGGTCTCCACT-16', 'AAAGATGCAAGTCTGT-9', 'AAAGTAGAGAGTCGGT-2',
       'AAAGTAGGTTCGCGAC-9', 'AACACGTAGTGAATTG-7', 'AACATGCGTATGGTCG-19',
       'AACGTTGCAGCTGTTA-15', 'AACTCAGCATTGGGCC-8', 'AACTTTCCATTTCACT-9',
       'AAGCCGCGTACGCTGC-8',
       ...
       'TAGTGGTGTGAGGGAG-10', 'TCGCGTTAGTTACGGG-15', 'TGCTGCTAGCGAAGGG-10',
       'TGCTGCTGTCAGAGGT-1', 'TGCTGCTTCAGCTGGC-7', 'TGTGTTTGTCACAAGG-4',
       'TTCTCAATCTCCTATA-2', 'TTGACTTCACTATCTT-15', 'TTGACTTTCCGAAGAG-12',
       'TTGCCGTCATCCGGGT-2'],
      dtype='object', length=357246)

In [11]:
# --- 3. Filter adata_singlets to retain only matching rows ---
# We use the .isin() method on adata_singlets.obs.index to create a boolean mask
# which is then used to subset adata_singlets.
adata_singlets_filtered = adata_singlets[adata_singlets.index.isin(desired_indices)].copy()


In [12]:
adata_singlets_filtered

,BEX3_1_ctl,BEX3_2_ctl,BEX3_3_ctl,BEX3_4_ctl,FN1_1_1_trunc_ctl,FN1_1_2_trunc_ctl,FN1_1_3_trunc_ctl,FN1_2_1_trunc_ctl,FN1_2_3_trunc_ctl,FN1_pro_1_1_ctl,...,chrX_67546350_G_C_AR_polyalanine_3,chrX_67546350_G_C_AR_polyalanine_4,chrX_67546350_G_C_AR_polyalanine_ref_1,chrX_67546350_G_C_AR_polyalanine_ref_2,chrX_67546351_C_T_AR_polyalanine_1,chrX_67546351_C_T_AR_polyalanine_2,chrX_67546351_C_T_AR_polyalanine_3,chrX_67546351_C_T_AR_polyalanine_4,chrX_67546351_C_T_AR_polyalanine_ref_1,chrX_67546351_C_T_AR_polyalanine_ref_2
AAACCTGAGAAGGACA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCTGAGACTAGAT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCTGAGCGTTCCG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCTGAGGAGTACC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCTGAGGCGTACA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GTTGTCTTCCCGTAAA-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GTTGTGCAGCAGGGGT-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GTTGTGCAGGGATAGT-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GTTGTGGGTATTGTCT-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
adata_singlets_filtered_transposed = adata_singlets_filtered.T

In [14]:
adata_singlets_filtered_transposed

,AAACCTGAGAAGGACA-1,AAACCTGAGACTAGAT-1,AAACCTGAGCGTTCCG-1,AAACCTGAGGAGTACC-1,AAACCTGAGGCGTACA-1,AAACCTGAGTAGCGGT-1,AAACCTGAGTATGACA-1,AAACCTGAGTGAACGC-1,AAACCTGAGTTTCCTT-1,AAACCTGCAACCGCCA-1,...,GTTGTCCGTCGACGGT-19,GTTGTCCGTCTCGGTT-19,GTTGTCCGTGCTCTTA-19,GTTGTCCGTGGGGAAT-19,GTTGTCTTCACAGATT-19,GTTGTCTTCCCGTAAA-19,GTTGTGCAGCAGGGGT-19,GTTGTGCAGGGATAGT-19,GTTGTGGGTATTGTCT-19,GTTGTGGGTCCCTAGT-19
BEX3_1_ctl,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BEX3_2_ctl,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BEX3_3_ctl,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BEX3_4_ctl,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FN1_1_1_trunc_ctl,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrX_67546351_C_T_AR_polyalanine_2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
chrX_67546351_C_T_AR_polyalanine_3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
chrX_67546351_C_T_AR_polyalanine_4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
chrX_67546351_C_T_AR_polyalanine_ref_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
print("\nFiltered adata_singlets.obs index (should match original adata.obs index):")
print(adata_singlets_filtered.index)


Filtered adata_singlets.obs index (should match original adata.obs index):
Index(['AAACCTGAGAAGGACA-1', 'AAACCTGAGACTAGAT-1', 'AAACCTGAGCGTTCCG-1',
       'AAACCTGAGGAGTACC-1', 'AAACCTGAGGCGTACA-1', 'AAACCTGAGTAGCGGT-1',
       'AAACCTGAGTATGACA-1', 'AAACCTGAGTGAACGC-1', 'AAACCTGAGTTTCCTT-1',
       'AAACCTGCAACCGCCA-1',
       ...
       'GTTGTCCGTCGACGGT-19', 'GTTGTCCGTCTCGGTT-19', 'GTTGTCCGTGCTCTTA-19',
       'GTTGTCCGTGGGGAAT-19', 'GTTGTCTTCACAGATT-19', 'GTTGTCTTCCCGTAAA-19',
       'GTTGTGCAGCAGGGGT-19', 'GTTGTGCAGGGATAGT-19', 'GTTGTGGGTATTGTCT-19',
       'GTTGTGGGTCCCTAGT-19'],
      dtype='object', length=357246)


In [17]:
print("\nShape of original adata_singlets:", adata_singlets.shape)
print("Shape of filtered adata_singlets:", adata_singlets_filtered.shape)


Shape of original adata_singlets: (471396, 4889)
Shape of filtered adata_singlets: (357246, 4889)


In [15]:
adata_singlets_filtered_transposed.to_pickle('/project/GCRB/Hon_lab/s215194/Single_Cell/EP1/_all_lanes_combined/aggr_dataframe/aggr_combined_df_full_clones_removed.pkl')